In [11]:
#Load the YAML configuration 
%load_ext autoreload
%autoreload 2
from eeg_toolkit import (
    load_config,
    extract_mean_band_power,
)
cfg = load_config('../../configs/eye_eeg_simul.yaml')
cfg_tfr = load_config('../../configs/tfr_analysis.yaml')
print("Setup OK")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Setup OK


In [3]:
#Here you are going to select the ROIs in the three dimension that you are interested to export
rois = {
    "frontal_mid":    ["Fz", "F3", "F4"],
    "central_mid":    ["Cz", "C3", "C4"],
    "posterior_mid":   ["Pz", "Oz"],
    "posterior_left":  ["P3", "P7", "O1"],
    "posterior_right": ["P4", "P8", "O2"],
}

freq_bands = {
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
}

time_windows = {
    "early":  (0.1, 0.3),
    "mid":    (0.3, 0.5),
    "late":   (0.5, 0.8),
}

In [7]:
#Here we are constructing the data frame that is going to store the values of all the data you wanted to extract to 
#later export it to a csv and run some statistical testings.
df = extract_mean_band_power(
    cfg, cfg_tfr, "cue", rois, freq_bands, time_windows,
    conditions=["spatial", "symbolic"]
)
df.head(10)

[find_subjects] excluded 8 subject(s): ['subj13', 'subj15', 'subj16', 'subj17', 'subj2', 'subj20', 'subj28', 'subj7']
Extracted 2700 rows: 30 subjects × 2 conditions × 5 ROIs × 3 bands × 3 time windows


,subject,condition,roi,channels,freq_band,fmin,fmax,time_window,tmin,tmax,mean_power
0,subj3,spatial,frontal_mid,"Fz, F3, F4",theta,4,8,early,0.1,0.3,0.090832
1,subj3,spatial,frontal_mid,"Fz, F3, F4",theta,4,8,mid,0.3,0.5,0.013796
2,subj3,spatial,frontal_mid,"Fz, F3, F4",theta,4,8,late,0.5,0.8,-0.070860
3,subj3,spatial,frontal_mid,"Fz, F3, F4",alpha,8,13,early,0.1,0.3,0.002950
4,subj3,spatial,frontal_mid,"Fz, F3, F4",alpha,8,13,mid,0.3,0.5,-0.145667
5,subj3,spatial,frontal_mid,"Fz, F3, F4",alpha,8,13,late,0.5,0.8,-0.304441
6,subj3,spatial,frontal_mid,"Fz, F3, F4",beta,13,30,early,0.1,0.3,-0.028272
7,subj3,spatial,frontal_mid,"Fz, F3, F4",beta,13,30,mid,0.3,0.5,-0.092859
8,subj3,spatial,frontal_mid,"Fz, F3, F4",beta,13,30,late,0.5,0.8,-0.097690
9,subj3,spatial,central_mid,"Cz, C3, C4",theta,4,8,early,0.1,0.3,0.057996


In [5]:
from eeg_toolkit.io import get_analysis_dir
out_dir = get_analysis_dir(cfg) / "group_results" / "tfr" / "stats"
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "mean_band_power_cue.csv"
df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

Saved: C:\Users\juapa\OneDrive\Documentos\Proyecto doctorado\Experimentos_proyecto_tesis\Experimentos Eye Tracker\analisis\analisis_eeg\group_results\tfr\stats\mean_band_power_cue.csv


In [8]:
#This is a way to do the standard comparison spatial vs symbolic in my design with all the possible variables that
# were selected on the previous code, this is just a way to do it in python, but with the csv it can be done in any statistical 
#software
from eeg_toolkit import run_tfr_paired_tests

stats_df = run_tfr_paired_tests(df, contrast=("spatial", "symbolic"))
stats_df

Paired tests: 45 comparisons, 0 significant after FDR


,roi,freq_band,fmin,fmax,time_window,tmin,tmax,n,mean_a,mean_b,mean_diff,cohens_d,t_stat,p_ttest,w_stat,p_wilcoxon,p_fdr,sig_fdr,contrast
0,central_mid,alpha,8,13,early,0.1,0.3,30,0.008721,0.006157,0.002564,0.046993,0.257393,0.798693,206.0,0.597808,0.954240,False,spatial_vs_symbolic
1,central_mid,alpha,8,13,late,0.5,0.8,30,-0.147528,-0.148546,0.001018,0.015785,0.086460,0.931695,228.0,0.935399,0.954240,False,spatial_vs_symbolic
2,central_mid,alpha,8,13,mid,0.3,0.5,30,-0.054540,-0.062187,0.007646,0.123372,0.675735,0.504562,185.0,0.338742,0.870281,False,spatial_vs_symbolic
3,frontal_mid,alpha,8,13,early,0.1,0.3,30,-0.028222,-0.042947,0.014725,0.220216,1.206172,0.237500,176.0,0.253436,0.822115,False,spatial_vs_symbolic
4,frontal_mid,alpha,8,13,late,0.5,0.8,30,-0.155903,-0.171779,0.015876,0.230414,1.262029,0.216994,170.0,0.205410,0.813729,False,spatial_vs_symbolic
5,frontal_mid,alpha,8,13,mid,0.3,0.5,30,-0.097324,-0.119878,0.022554,0.293209,1.605974,0.119115,156.0,0.119077,0.724268,False,spatial_vs_symbolic
6,posterior_left,alpha,8,13,early,0.1,0.3,30,-0.074771,-0.086310,0.011540,0.171061,0.936941,0.356527,182.0,0.308521,0.832743,False,spatial_vs_symbolic
7,posterior_left,alpha,8,13,late,0.5,0.8,30,-0.245114,-0.247047,0.001933,0.030378,0.166389,0.869006,223.0,0.855272,0.954240,False,spatial_vs_symbolic
8,posterior_left,alpha,8,13,mid,0.3,0.5,30,-0.176722,-0.185997,0.009275,0.124581,0.682357,0.500428,214.0,0.715133,0.870281,False,spatial_vs_symbolic
9,posterior_mid,alpha,8,13,early,0.1,0.3,30,-0.092993,-0.107643,0.014649,0.259122,1.419267,0.166481,163.0,0.157948,0.749165,False,spatial_vs_symbolic


In [9]:
# Only theta at posterior ROIs, just to show how to filter the df and apply the same paired test. 
df_post_theta = df[
    (df["freq_band"] == "theta") &
    (df["roi"].isin(["posterior_mid", "posterior_left", "posterior_right"]))
]
run_tfr_paired_tests(df_post_theta, contrast=("spatial", "symbolic"))

Paired tests: 9 comparisons, 0 significant after FDR


,roi,freq_band,fmin,fmax,time_window,tmin,tmax,n,mean_a,mean_b,mean_diff,cohens_d,t_stat,p_ttest,w_stat,p_wilcoxon,p_fdr,sig_fdr,contrast
0,posterior_left,theta,4,8,early,0.1,0.3,30,0.075603,0.066414,0.009189,0.153581,0.841200,0.407119,187.0,0.359878,0.776224,False,spatial_vs_symbolic
1,posterior_left,theta,4,8,late,0.5,0.8,30,-0.051709,-0.047410,-0.004299,-0.067244,-0.368312,0.715315,212.0,0.685047,0.776224,False,spatial_vs_symbolic
2,posterior_left,theta,4,8,mid,0.3,0.5,30,0.018898,0.022358,-0.003460,-0.052382,-0.286906,0.776224,219.0,0.792159,0.776224,False,spatial_vs_symbolic
3,posterior_mid,theta,4,8,early,0.1,0.3,30,0.062542,0.057347,0.005194,0.102131,0.559395,0.580188,186.0,0.349211,0.776224,False,spatial_vs_symbolic
4,posterior_mid,theta,4,8,late,0.5,0.8,30,-0.052140,-0.045473,-0.006667,-0.107292,-0.587664,0.561306,221.0,0.823577,0.776224,False,spatial_vs_symbolic
5,posterior_mid,theta,4,8,mid,0.3,0.5,30,0.032962,0.025286,0.007676,0.124281,0.680715,0.501451,175.0,0.244946,0.776224,False,spatial_vs_symbolic
6,posterior_right,theta,4,8,early,0.1,0.3,30,0.060765,0.050080,0.010685,0.154259,0.844912,0.405077,213.0,0.700033,0.776224,False,spatial_vs_symbolic
7,posterior_right,theta,4,8,late,0.5,0.8,30,-0.093079,-0.089175,-0.003904,-0.058304,-0.319343,0.751756,228.0,0.935399,0.776224,False,spatial_vs_symbolic
8,posterior_right,theta,4,8,mid,0.3,0.5,30,-0.015772,-0.023456,0.007685,0.103684,0.567903,0.574472,191.0,0.404495,0.776224,False,spatial_vs_symbolic
